<a href="https://colab.research.google.com/github/vchandraiitk/agentic-ai/blob/main/lcel/LCEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Embedding Techniques Using HuggingFace

In [1]:
pip --quiet install dotenv langchain_community langchain_groq langchain_openai fastapi langserve sse_starlette pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.2/69.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.8/130.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.3 MB/s eta 0:00:00


In [16]:
import os
from dotenv import load_dotenv
from google.colab import userdata
load_dotenv()

os.environ['OPENAI_API_KEY']=userdata.get("OPENAI_API_KEY")
os.environ['GROQ_API_KEY']=userdata.get("GROQ_API_KEY")
grok_key=userdata.get("NGROK_AUTH")

# ## Langsmith Tracking
# os.environ["LANGCHAIN_API_KEY"]=userdata.get("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_TRACING_V2"]="true"
# os.environ["LANGCHAIN_PROJECT"]=userdata.get("LANGCHAIN_PROJECT")

In [4]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
model = ChatGroq(model="Gemma2-9b-It", groq_api_key=os.environ['GROQ_API_KEY'])
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7c08e2c5fd90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c08e279b790>, model_name='Gemma2-9b-It', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.messages import HumanMessage, SystemMessage
message=[
    SystemMessage(content="Translate the following from English to French"),
    HumanMessage(content="Hello how are you?")
    ]
result = model.invoke(message)
result

AIMessage(content="Bonjour, comment allez-vous ? \n\n\nHere's a breakdown:\n\n* **Bonjour** - Hello\n* **comment** - how\n* **allez-vous** - are you (formal, polite)\n\n\nLet me know if you'd like to learn more French phrases!\n", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 62, 'prompt_tokens': 21, 'total_tokens': 83, 'completion_time': 0.112727273, 'prompt_time': 0.002468401, 'queue_time': 0.09629314300000001, 'total_time': 0.115195674}, 'model_name': 'Gemma2-9b-It', 'system_fingerprint': 'fp_10c08bf97d', 'finish_reason': 'stop', 'logprobs': None}, id='run--8ca2cd12-e9e8-49d8-b7ef-a8ffeb3c7687-0', usage_metadata={'input_tokens': 21, 'output_tokens': 62, 'total_tokens': 83})

In [6]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
output = parser.invoke(result)
print(output)

Bonjour, comment allez-vous ? 


Here's a breakdown:

* **Bonjour** - Hello
* **comment** - how
* **allez-vous** - are you (formal, polite)


Let me know if you'd like to learn more French phrases!



In [7]:
chain = model|parser
chain.invoke(message)

'**Bonjour, comment allez-vous ?** \n\n\nThis is the most common and formal way to say "Hello, how are you?" in French. \n\nHere are some other options:\n\n* **Salut, ça va ?** (Informal, more casual)\n* **Coucou, comment vas-tu ?** (Very informal, used with friends or family)\n\n\nLet me know if you have any other phrases you\'d like translated!\n'

In [8]:
from langchain_core.prompts import ChatPromptTemplate
generic_template = "Translate sentence to {language} language"
prompt = ChatPromptTemplate.from_messages([("system",generic_template), ("user", "{text}")])

In [9]:
prompt_result = prompt.invoke({"language":"French", "text":"We are learning agenticAI in day-4"})

In [10]:
chain = prompt|model|parser
chain.invoke({"language":"French", "text":"We are learning agenticAI in day-4"})

"We are learning agenticAI on day 4. \n\n\nHere's a breakdown:\n\n* **We are learning** translates to **Nous apprenons**.\n* **agenticAI** is left as is since it's a specific term.\n* **in day-4** translates to **le 4e jour**. \n\n\nLet me know if you have any other sentences you'd like me to translate!\n"

In [18]:
!ngrok config add-authtoken grok_key

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [63]:
# generic_template = "Translate sentence to {language} language"
# prompt = ChatPromptTemplate.from_messages([("system",generic_template), ("user", "{text}")])

In [12]:
from langserve import add_routes
from fastapi import FastAPI
from sse_starlette import EventSourceResponse
from pyngrok import ngrok
app = FastAPI(title="langserve app",
              version="1.0",
              description="Doing my helloworld programming")
# Add your chain
add_routes(app, chain, path="/chain")

# # Optional: Add a custom /ping to verify app is running
# @app.get("/ping")
# def ping():
#     return {"status": "ok"}

In [13]:
if __name__=="__main__":
    import uvicorn
    import nest_asyncio
    nest_asyncio.apply()

    # Expose port 8000
    public_url = ngrok.connect(8000)
    print(f"🚀 Public URL: {public_url}")

    # Run server
    uvicorn.run(app, host="0.0.0.0", port=8000)


INFO:     Started server process [538]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


🚀 Public URL: NgrokTunnel: "https://a9c7-34-68-212-124.ngrok-free.app" -> "http://localhost:8000"

     __          ___      .__   __.   _______      _______. _______ .______     ____    ____  _______
    |  |        /   \     |  \ |  |  /  _____|    /       ||   ____||   _  \    \   \  /   / |   ____|
    |  |       /  ^  \    |   \|  | |  |  __     |   (----`|  |__   |  |_)  |    \   \/   /  |  |__
    |  |      /  /_\  \   |  . `  | |  | |_ |     \   \    |   __|  |      /      \      /   |   __|
    |  `----./  _____  \  |  |\   | |  |__| | .----)   |   |  |____ |  |\  \----.  \    /    |  |____
    |_______/__/     \__\ |__| \__|  \______| |_______/    |_______|| _| `._____|   \__/     |_______|
    
LANGSERVE: Playground for chain "/chain/" is live at:
LANGSERVE:  │
LANGSERVE:  └──> /chain/playground/
LANGSERVE:
LANGSERVE: See all available routes at /docs/
INFO:     49.207.212.149:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     49.207.212.149:0 - "GET /favicon.ico HTTP/1.1" 404 Not

INFO:     Shutting down
INFO:     Finished server process [538]
